# 01 — Get and clean the data

We download **NBA Draft Combine** body measurements (2000 → today) with `nba_api`,
keep one row per player, convert to metric units and add a few engineered features.

Output: `data/processed/players.csv`

In [1]:
import time
from pathlib import Path

import pandas as pd
import truststore

truststore.inject_into_ssl()
from nba_api.stats.endpoints import draftcombineplayeranthro

RAW = Path("../data/raw/combine_raw.csv")
PROCESSED = Path("../data/processed/players.csv")

## 1. Download

One API call per season, with a short pause between calls. The result is cached in `data/raw/`, so delete that file to re-download.

`truststore` makes Python use the Windows certificate store, which is needed on networks that inspect HTTPS traffic.

In [2]:
def fetch_season(season):
    df = draftcombineplayeranthro.DraftCombinePlayerAnthro(season_year=season, timeout=60).get_data_frames()[0]
    time.sleep(1)
    return df.assign(SEASON=season)


if RAW.exists():
    raw = pd.read_csv(RAW)
else:
    seasons = [f"{y}-{str(y + 1)[-2:]}" for y in range(2000, 2026)]
    raw = pd.concat([fetch_season(s) for s in seasons], ignore_index=True)
    RAW.parent.mkdir(parents=True, exist_ok=True)
    raw.to_csv(RAW, index=False)

raw.head()

,TEMP_PLAYER_ID,PLAYER_ID,FIRST_NAME,LAST_NAME,PLAYER_NAME,POSITION,HEIGHT_WO_SHOES,HEIGHT_WO_SHOES_FT_IN,HEIGHT_W_SHOES,HEIGHT_W_SHOES_FT_IN,WEIGHT,WINGSPAN,WINGSPAN_FT_IN,STANDING_REACH,STANDING_REACH_FT_IN,BODY_FAT_PCT,HAND_LENGTH,HAND_WIDTH,SEASON
0,NaN,2124,Malik,Allen,Malik Allen,PF-C,80.25,6' 8.25'',NaN,NaN,271.0,86.5,7' 2.5'',109.0,9' 1'',NaN,NaN,NaN,2000-01
1,NaN,12019,Harold,Arceneaux,Harold Arceneaux,SG-SF,76.50,6' 4.5'',NaN,NaN,219.0,80.5,6' 8.5'',103.0,8' 7'',NaN,NaN,NaN,2000-01
2,NaN,12020,Lamont,Barnes,Lamont Barnes,PF-C,80.50,6' 8.5'',NaN,NaN,235.5,87.5,7' 3.5'',108.0,9' 0'',NaN,NaN,NaN,2000-01
3,NaN,12131,Mario,Bland,Mario Bland,PF,77.50,6' 5.5'',NaN,NaN,287.0,84.0,7' 0'',103.0,8' 7'',NaN,NaN,NaN,2000-01
4,NaN,2056,Primoz,Brezec,Primoz Brezec,C,84.75,7' 0.75'',NaN,NaN,243.0,86.0,7' 2'',110.0,9' 2'',NaN,NaN,NaN,2000-01


## 2. Clean

* **Label:** positions like `"SG-SF"` become their first listed position (`"SG"`).
* Drop players without a position, height or weight.
* Players who attended the combine twice: keep their latest measurement.
* Convert inches → cm and pounds → kg.

In [3]:
raw["POSITION"].value_counts()

POSITION
PF       311
PG       285
SG       265
SF       230
C        162
SG-SF     95
PF-C      83
PG-SG     79
SF-PF     75
C-PF      62
PF-SF     55
SF-SG     48
SG-PG     40
Name: count, dtype: int64

In [4]:
df = raw.copy()
df["position"] = df["POSITION"].str.split("-").str[0].str.strip()

num_cols = ["HEIGHT_WO_SHOES", "WEIGHT", "WINGSPAN", "STANDING_REACH", "BODY_FAT_PCT", "HAND_LENGTH", "HAND_WIDTH"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

df = df[df["position"].isin(["PG", "SG", "SF", "PF", "C"])]
df = df.dropna(subset=["HEIGHT_WO_SHOES", "WEIGHT"])
df = df.sort_values("SEASON").drop_duplicates("PLAYER_ID", keep="last")

IN, LB = 2.54, 0.4536
players = pd.DataFrame({
    "player_id": df["PLAYER_ID"],
    "player": df["PLAYER_NAME"],
    "season": df["SEASON"],
    "height_cm": df["HEIGHT_WO_SHOES"] * IN,
    "weight_kg": df["WEIGHT"] * LB,
    "wingspan_cm": df["WINGSPAN"] * IN,
    "reach_cm": df["STANDING_REACH"] * IN,
    "body_fat_pct": df["BODY_FAT_PCT"],
    "hand_length_cm": df["HAND_LENGTH"] * IN,
    "hand_width_cm": df["HAND_WIDTH"] * IN,
    "position": df["position"],
})

## 3. Feature engineering

New columns computed from existing ones, which can carry extra signal:

* **ape index** = wingspan − height (long arms for your height)
* **reach minus height**
* **BMI** = weight / height² (how heavy for your height)

In [5]:
players["ape_index_cm"] = players["wingspan_cm"] - players["height_cm"]
players["reach_minus_height_cm"] = players["reach_cm"] - players["height_cm"]
players["bmi"] = players["weight_kg"] / (players["height_cm"] / 100) ** 2

players = players.round(2).reset_index(drop=True)
PROCESSED.parent.mkdir(parents=True, exist_ok=True)
players.to_csv(PROCESSED, index=False)
players.head()

,player_id,player,season,height_cm,weight_kg,wingspan_cm,reach_cm,body_fat_pct,hand_length_cm,hand_width_cm,position,ape_index_cm,reach_minus_height_cm,bmi
0,2124,Malik Allen,2000-01,203.84,122.93,219.71,276.86,NaN,NaN,NaN,PF,15.88,73.03,29.59
1,2061,Dan Langhi,2000-01,205.74,89.59,203.20,264.16,NaN,NaN,NaN,SF,-2.54,58.42,21.16
2,2090,Justin Love,2000-01,186.69,93.21,194.31,247.65,NaN,NaN,NaN,PG,7.62,60.96,26.75
3,2058,Mark Madsen,2000-01,203.20,107.28,214.63,265.43,NaN,NaN,NaN,PF,11.43,62.23,25.98
4,2082,Dan McClintock,2000-01,210.82,113.40,217.17,271.78,NaN,NaN,NaN,C,6.35,60.96,25.51


## 4. Sanity checks

How many players per position (class balance), and how much data is missing per column.

In [6]:
players["position"].value_counts().reindex(["PG", "SG", "SF", "PF", "C"])

position
PG    347
SG    378
SF    328
PF    425
C     202
Name: count, dtype: int64

In [7]:
players.isna().mean().mul(100).round(1).rename("% missing")

player_id                 0.0
player                    0.0
season                    0.0
height_cm                 0.0
weight_kg                 0.0
wingspan_cm               0.0
reach_cm                  0.1
body_fat_pct             26.4
hand_length_cm           43.5
hand_width_cm            43.5
position                  0.0
ape_index_cm              0.0
reach_minus_height_cm     0.1
bmi                       0.0
Name: % missing, dtype: float64